In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
#

In [ ]:
# load the cleaned ATS pairs dataset from Notebook 1
df = pd.read_csv('cleaned_resumeJD_pairs.csv')
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['match_label'].value_counts())
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_resumeJD_pairs.csv'

In [ ]:
print('loading the bert model')
model = SentenceTransformer('all-mpnet-base-v2')

print("Model Loaded")
print(f" Model: all-mpnet-base-v2")
print(f" Paramaeters: 22.7M")
print(f" Embed dims: 768")
print(f" Max tokens: {model.max_seq_length}")

loading the bert model


NameError: name 'SentenceTransformer' is not defined

In [ ]:
# Part 2: Understanding Embeddings
texts = [
    "Python Developer with Django experience",
    "Python Software Engineer, Django REST APIs",
    "Frontend Developer React and JavaScript",
    "Excecutive Chef with Michelin experience",
    "Data Scientist, Machine Learning, Pytorch",
]

embeddings = model.encode(texts)
sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize = (8, 6))
sns.heatmap(sim_matrix, annot = True, fmt = '.2f', cmap = 'RdYlGn',
            xticklabels = [t[:25] for t in texts],
            yticklabels = [t[:25] for t in texts])

plt.title('Semantic Similarity - BERT Embeddings', fontsize = 14, fontweight = 'bold')
plt.tight_layout()
plt.show()

            )

In [ ]:
# generating resume and jd embeddings
# we embed BOTH resume and job description seprately
print("Generating resume embedding...")
resume_embeddings = model.encode(
    df['reusme_text'].tolist(),
    batch_size = 32,
    show_progress_bar = True,
    convert_to_numpy = True
)

print("\nGenerating job description embeddings...")
jd_embeddings = model.encode(
    df['job_description'].tolist(),
    batch_size = 32,
    show_progress_bar = True,
    convert_to_numpy = True
)
print(f"\nResume embeddings shape: {resume_embeddings.shape}")
print(f"JD embeddings shape: {jd_embeddings.shape}")
print(f"Each text -> {resume_embeddings.shape[1]} - dimensional vector")

In [ ]:
# compute cosine similarity for each pair
# for each row: similarity between that resume and its paired JD

pair_similarity = []
for i in range(len(resume_embeddings)):
  sim = cosine_similarity([resume_embeddings[i], [jd_embeddings[i]]])[0][0]
  pair_similarities.append(sim)

  df['bert_similarity'] = pair_similarities

  print("Cosine similarity computed for all pairs.")
  print(df[['match_label', 'match_score', 'bert_similarity']].head(10))


In [ ]:
#Base model Performance: How Good Is BERT Out of the Box ?

mae = mean_absolute_error(df['match_score'], df['bert_similarity'])
rmse = np.sqrt(mean_squared_error(df['match_score'], df['bert_similarity']))

print("Base BERT Model Performance (before fine- tuning)")
print("=" * 50)
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

In [ ]:
# Scatter: predicted vs ground truth
plt.figure(figsize = (8,6))
colors = df['match_label'].map({'low': 'red', 'medium':'orange', 'high':'green'})
plt.scatter(df['match_score'], df['bert_similarity'], c= colors, alpha = 0.6)
plt.plot([0,1], [0,1], 'k--', label = 'Perfect prediction')


plt.xlabel('Ground Truth match_score')
plt.ylabel('BERT Cosine Similarity')
plt.title('Base BERT: Predicted vs Actual Match Score', fontweight = 'bold')

# Legend
from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(color='green', label='high'),
    Patch(color='orange', label='medium')
    Patch(color='red', label='low'),
    plt.line2D([0], [0], color = 'black', linestyle = '--', label = 'Perfect')
])
plt.tight_layout()
plt.show()


In [ ]:
# Similarity Distribution by Label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram: Best similarity score grouped by label
for label, color in [('low', 'blue'), ('medium', 'orange'), ('high', 'green')]:
    subset = df[df['match_label'] == label]['similarity']
    axes[0].hist(subset, bins=20, alpha=0.6, label=label, color=color)

axes[0].set_xlabel('Best Similarity')
axes[0].set_ylabel('Count')
axes[0].set_title('Similarity Distribution by Label')
axes[0].legend()

# Box plot
df.boxplot(column='best_similarity', by='match_label', positions=[0, 1, 2], ax=axes[1])
axes[1].set_title('Best Similarity per Label (Box Plot)')
axes[1].set_xlabel('Match Label')
axes[1].set_ylabel('Cosine Similarity')
plt.suptitle('')  # Remove automatic boxplot title
plt.tight_layout()

In [ ]:
import pickle

with open('resume_embeddings.pkl', 'wb') as f:
    pickle.dump({
        'resume_embeddings': resume_embeddings,
        'jd_embeddings': jd_embeddings,
        'match_scores': df['match_score'].tolist(),
        'match_labels': df['match_label'].tolist(),
    }, f)

print("Saved: resume_embeddings.pkl")
print(f"  resume_embeddings: {resume_embeddings.shape}")
print(f"  jd_embeddings: {jd_embeddings.shape}")
print()
print("These will be loaded for fine-tuning.")